# Values — Polyglot Reference

Side-by-side syntax for **literals, bindings, and value semantics** across six languages.
Each row is one atomic sub-feature — scan across to see the equivalents.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

Cells hold inline code only. Anything that needs more than a one-liner is expanded in the **Notes** section below the tables.

## Literals

### How values are stored

Every literal in these six languages compiles to one of three storage modes. The *syntax* of `42` looks identical across the row in the table below — the *representation* in memory does not.

![Three storage modes: primitive, tagged/object, boxed reference](https://raw.githubusercontent.com/schemabotview/polyglot/main/img/values-storage-modes.svg)

- **Java** — `int`/`long`/`double` are primitives (stack/register, no heap). `Integer`/`Long`/`Double` are boxed wrappers. Autoboxing converts between them implicitly, but with cost — `Integer.valueOf(1000) == Integer.valueOf(1000)` is `false` because each call allocates a new wrapper.
- **Scala / Kotlin** — present a unified `Int` type. The compiler emits a JVM primitive when possible and boxes only when required: generics (`List[Int]`), nullable types (`Int?` in Kotlin, `Option[Int]` in Scala), pattern matching, etc. You write one type; the JVM sees two.
- **JavaScript / TypeScript** — V8 uses **SMI** tagging: small integers are stored inline as tagged pointers — no heap allocation. Doubles, larger ints, and all `BigInt` go on the heap. This is an implementation detail, not part of the language spec.
- **Python** — every value is a `PyObject*` reference. Even `42` is a full heap object with a refcount and type pointer. Small integers `-5..256` are pre-allocated once at interpreter startup and reused — so `42 is 42` is always `True`, but `1000 is 1000` may be `False`.

Keep this picture in mind while reading the table below — `42` is one syntax, three storage stories.

### Syntax table

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| integer | `42` | `42` | `42` | `42` | `42` | `42` |
| long / bigint | `42L` | `42L` | `42L` | `42n` | `42n` | `42` *(arbitrary precision)* |
| float (32-bit) | `3.14f` | `3.14f` | `3.14f` | — *(all 64-bit)* | — | — |
| double | `3.14` | `3.14` | `3.14` | `3.14` | `3.14` | `3.14` |
| hex | `0xFF` | `0xFF` | `0xFF` | `0xFF` | `0xFF` | `0xFF` |
| binary | `0b1010` | — | `0b1010` | `0b1010` | `0b1010` | `0b1010` |
| underscore separator | `1_000_000` | `1_000_000` *(2.13+)* | `1_000_000` | `1_000_000` | `1_000_000` | `1_000_000` |
| boolean | `true` / `false` | `true` / `false` | `true` / `false` | `true` / `false` | `true` / `false` | `True` / `False` |
| null / none | `null` | `null` | `null` | `null`, `undefined` | `null`, `undefined` | `None` |
| char | `'a'` | `'a'` | `'a'` | — *(strings only)* | — | — |
| string | `"hi"` | `"hi"` | `"hi"` | `"hi"` or `'hi'` | `"hi"` or `'hi'` | `"hi"` or `'hi'` |
| multiline string | `"""hi"""` *(15+)* | `"""hi"""` | `"""hi"""` | `` `hi` `` | `` `hi` `` | `"""hi"""` |
| interpolation | `"%s".formatted(x)` | `s"$x"` | `"$x"` | `` `${x}` `` | `` `${x}` `` | `f"{x}"` |

## Bindings

Immutability and initialization timing — the *value semantics* of a binding. (Type annotations, type inference, and scoping rules live in **02 — Names, Types & Generics**.)

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| immutable | `final int x = 1;` | `val x = 1` | `val x = 1` | `const x = 1` | `const x = 1` | `X = 1` *(convention only)* |
| mutable | `int x = 1;` | `var x = 1` | `var x = 1` | `let x = 1` | `let x = 1` | `x = 1` |
| compile-time const | `static final int X = 1;` | `final val X = 1` | `const val X = 1` | — *(runtime only)* | — *(runtime only)* | — |
| lazy init | — | `lazy val x = ...` | `val x by lazy { ... }` | — | — | — |

## Equality & Identity

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| value equality | `a.equals(b)` | `a == b` | `a == b` | `a === b` | `a === b` | `a == b` |
| reference identity | `a == b` | `a eq b` | `a === b` | `a === b` *(objects)* | `a === b` | `a is b` |
| coerced equality | — | — | — | `a == b` | `a == b` | — |
| null-safe equality | `Objects.equals(a,b)` | `a == b` *(safe)* | `a == b` *(safe)* | `a === b` *(safe)* | `a === b` *(safe)* | `a == b` *(safe)* |
| check for null | `a == null` | `a == null` | `a == null` | `a == null` *(both)* | `a == null` | `a is None` |
| hash | `a.hashCode()` | `a.hashCode` | `a.hashCode()` | — *(no built-in)* | — | `hash(a)` |

## Notes — when a cell isn't enough

**Java `==` vs `.equals()`.** For objects (including `String`), `==` compares references, not values. Use `.equals()` for value equality. Primitives use `==` directly. This is the single most common Java footgun.

**Scala `==` inverts Java.** `==` calls `.equals()` under the hood, so it's value equality even for objects. Use `eq` for reference identity. The Java default is the rare case.

**Kotlin keeps both explicit.** `==` translates to `.equals()` with null safety baked in (`null == null` is `true`, no NPE). `===` is reference identity. Mirror image of Java.

**JavaScript `null` vs `undefined`.** Two distinct "empty" values. `undefined` = declared but unassigned, or missing property, or function with no return. `null` = explicitly empty, set by the programmer. `==` treats them as equal to each other (and only each other); `===` does not. Idiom: `if (x == null)` catches both.

**Python `is None`.** Always use `is None` / `is not None`, never `== None`. `None` is a singleton, so identity is the correct check and is faster. Linters flag `== None`.

**String interpolation.** Scala `s"..."`, Kotlin `"..."`, Python `f"..."`, and JS/TS template literals all interpolate at the call site at compile/parse time. Java's `String.formatted` (15+) is `printf`-style — closer to a method call than true interpolation. Scala also has `f"..."` (typed format) and `raw"..."` (no escapes).

**`const` in JS/TS is shallow.** `const obj = { a: 1 }; obj.a = 2;` is legal — `const` prevents rebinding the name, not mutation of the value. Java's `final`, Scala/Kotlin's `val` have the same semantics. Only Kotlin's `const val` (and Java's `static final` with a primitive/`String` literal) are true compile-time constants.